In [ ]:
import numpy as np
import pandas as pd
from IPython.display import clear_output
import xgboost
import sklearn
import pylift
from annoy import AnnoyIndex
import random

In [ ]:
# from google.cloud import bigquery


In [ ]:
# client = bigquery.Client()

In [ ]:
%%bash
gsutil cp results.csv gs://uplift-criteo/

In [ ]:
data = pd.read_csv('train.csv')

In [ ]:
data.head()

In [ ]:
test    = data[data.treatment == 1]
control = data[data.treatment == 0]

In [ ]:
print(test.shape[0])
print(control.shape[0])

In [ ]:
f = 12
t = AnnoyIndex(f)  # Length of item vector that will be indexed
for i in range(test.shape[0]):
    if i % 50000 == 0:
        clear_output()
        print(i/test.shape[0])
    ix = test.index[i]
    v = test.loc[ix][:-4]
    t.add_item(ix, v)

In [ ]:
t.build(10) # 10 trees

In [ ]:
t.save('test_full.ann')

In [ ]:
# u = AnnoyIndex(f)
# u.load('test.ann') # super fast, will just mmap the file
# print(u.get_nns_by_item(0, 1000)) # will find the 1000 nearest neighbors

In [ ]:
control.head()

In [ ]:
matched_items = []
for row in range(control.shape[0]):
    d = t.get_nns_by_vector(control.iloc[row][:-4], 1, include_distances=True)
    control_ix      = control.index[row]
    test_ix         = d[0][0]
    dist            = d[1][0]
    test_outcome    = test.loc[test_ix][-3] 
    control_outcome = control.loc[control_ix][-3] 
    matched_items.append([control_ix, test_ix, control_outcome, test_outcome, dist])    
    if row % 50000 == 0:
        clear_output()
        print(row/control.shape[0])

In [ ]:
results = pd.DataFrame(matched_items)
results.columns = ['control_ix', 'test_ix', 'control_out', 'test_out', 'distance']

In [ ]:
results.index = results.control_ix

In [ ]:
results = pd.concat([results, control[control.columns[:-4]]], axis=1)

In [ ]:
test_slide = test.loc[list(results.test_ix)][test.columns[:-4]]

In [ ]:
test_slide.head()

In [ ]:
test_slide.columns = ['f0c','f1c','f2c','f3c','f4c','f5c','f6c','f7c','f8c','f9c','f10c','f11c']

In [ ]:
results.index = results.test_ix

In [ ]:
results = pd.concat([results, test_slide], axis=1)

In [ ]:
results.head()

Matching Diagnostics 

Now that we have performed matching, we need to perform some diagnostics. There are several test we will perform.

1. Classifier to T-test to insure we cannot predict treatment variable based on sample.
    - We know this is the case with the entire dataset, however as we used a nearest neighbor apporach to find distances, we may have introduced bias. 
2. The C2T test should allow us to measure any bias in best versus control, the next test will help us to understand variance. For this we will take the mean of each 

1. take mean of each column in test/control. Do hypothesis test to see if they are different
2. 

In [ ]:
results = pd.read_csv('results.csv')

In [ ]:
results.head()

In [ ]:
diff = np.array(results[['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']]) - results[['f0c', 'f1c', 'f2c', 'f3c', 'f4c', 'f5c', 'f6c', 'f7c', 'f8c', 'f9c', 'f10c', 'f11c']]

#diff.boxplot(column=['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11'])

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline  
plt.rcParams['figure.figsize'] = [15, 5]

diff = np.array(results[['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']]) - results[['f0c', 'f1c', 'f2c', 'f3c', 'f4c', 'f5c', 'f6c', 'f7c', 'f8c', 'f9c', 'f10c', 'f11c']]

diff.boxplot(column=['f0c', 'f1c', 'f2c', 'f3c', 'f4c', 'f5c', 'f6c', 'f7c', 'f8c', 'f9c', 'f10c', 'f11c'])

In [ ]:
# results['f11'] - results['f11c']

In [ ]:
# results[['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']].mean()

In [ ]:
# results[['f0c', 'f1c', 'f2c', 'f3c', 'f4c', 'f5c', 'f6c', 'f7c', 'f8c', 'f9c', 'f10c', 'f11c']].mean()

In [ ]:
from scipy.stats import ttest_ind

ttest_ind(results['f0'], results['f0c'])

In [ ]:
# low variance

In [ ]:
# any bias. I assume there will be.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.cross_validation import train_test_split
import numpy as np
from sklearn.metrics import hamming_loss
from scipy import stats

In [ ]:
clf=LogisticRegression()
loss=hamming_loss
bootstraps=300

c = results[['f0','f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11']]
c['treatment'] = 0

t = results[['f0c','f1c','f2c','f3c','f4c','f5c','f6c','f7c','f8c','f9c','f10c','f11c']]
t.columns = ['f0','f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11']
t['treatment'] = 1
            
c2test = pd.concat([t, c])
c2test = c2test.sample(frac=0.1)

X_train, X_test, y_train, y_test = train_test_split(c2test[['f0','f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11']], c2test['treatment'], test_size=0.33, random_state=35)

y_pred = clf.fit(X_train, y_train).predict(X_test)
emp_loss = loss(y_test, y_pred)
bs_losses = []
y_bar = np.mean(c2test['treatment'])

for b in range(100+1):
    clear_output()
    print(b/100.)
    y_random = np.random.binomial(1, y_bar, size=c2test['treatment'].shape[0])
    X_train, X_test, y_train, y_test = train_test_split(c2test[['f0','f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11']], y_random)
    y_pred_bs = clf.fit(X_train, y_train).predict(X_test)
    bs_losses += [loss(y_test, y_pred_bs)]
    
pc = stats.percentileofscore(sorted(bs_losses), emp_loss) / 100.
pvalue = pc if pc < y_bar else 1 - pc

In [ ]:
print("Loss of classifier trained on treatment variable of full dataset: %s" % emp_loss)
print("Average loss of classifers trained on random treatment variable: %s" % (np.mean(bs_losses)))

In [ ]:
stats.ttest_1samp(bs_losses, emp_loss)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.distplot(bs_losses, bins=50)
plt.axvline(emp_loss, color='r')
# plt.text(emp_loss+.1,0,'Treatment loss',rotation=90)
plt.axvline(np.mean(bs_losses), color='black')
plt.title("Distribution of Null Hypothesis and Treatment Loss")
plt.ylabel("Frequency")
plt.xlabel("Hamming Loss")

In [ ]:
# pd.DataFrame(results['test_ix'].value_counts()[:100])

In [ ]:
sns.distplot(results['test_ix'].value_counts())

In [ ]:
br = pd.DataFrame(results['test_ix'].value_counts()[:100]).reset_index()

In [ ]:
br.head()

In [ ]:
#sns.barplot(results['test_ix'].value_counts()[:100])
sns.barplot(x="index", y="test_ix", data=br, order=br['index'])

In [ ]:
br = pd.DataFrame(results['test_ix'].value_counts()).reset_index()

In [ ]:
br[br.test_ix ==1].shape[0] / br.shape[0]

In [ ]:
br = pd.DataFrame(results['test_ix'].value_counts()).reset_index()
results.index = results['test_ix']
sampled_results = results.loc[list(br[br.test_ix ==1]['index'])]

In [ ]:
sampled_results.head()

In [ ]:
diff = np.array(sampled_results[['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']]) - sampled_results[['f0c', 'f1c', 'f2c', 'f3c', 'f4c', 'f5c', 'f6c', 'f7c', 'f8c', 'f9c', 'f10c', 'f11c']]

diff.boxplot(column=['f0c', 'f1c', 'f2c', 'f3c', 'f4c', 'f5c', 'f6c', 'f7c', 'f8c', 'f9c', 'f10c', 'f11c'])

In [ ]:
clf=LogisticRegression()
loss=hamming_loss
bootstraps=300

c = sampled_results[['f0','f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11']]
c['treatment'] = 0

t = sampled_results[['f0c','f1c','f2c','f3c','f4c','f5c','f6c','f7c','f8c','f9c','f10c','f11c']]
t.columns = ['f0','f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11']
t['treatment'] = 1
            
c2test = pd.concat([t, c])
c2test = c2test.sample(frac=0.1)

X_train, X_test, y_train, y_test = train_test_split(c2test[['f0','f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11']], c2test['treatment'], test_size=0.33, random_state=35)

y_pred = clf.fit(X_train, y_train).predict(X_test)
emp_loss = loss(y_test, y_pred)
bs_losses = []
y_bar = np.mean(c2test['treatment'])

for b in range(100+1):
    clear_output()
    print(b/100.)
    y_random = np.random.binomial(1, y_bar, size=c2test['treatment'].shape[0])
    X_train, X_test, y_train, y_test = train_test_split(c2test[['f0','f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11']], y_random)
    y_pred_bs = clf.fit(X_train, y_train).predict(X_test)
    bs_losses += [loss(y_test, y_pred_bs)]
    
pc = stats.percentileofscore(sorted(bs_losses), emp_loss) / 100.
pvalue = pc if pc < y_bar else 1 - pc

print("Loss of classifier trained on treatment variable of full dataset: %s" % emp_loss)
print("Average loss of classifers trained on random treatment variable: %s" % (np.mean(bs_losses)))

In [ ]:
stats.ttest_1samp(bs_losses, emp_loss)

In [ ]:
sns.distplot(bs_losses, bins=50)
plt.axvline(emp_loss, color='r')
# plt.text(emp_loss+.1,0,'Treatment loss',rotation=90)
plt.axvline(np.mean(bs_losses), color='black')
plt.title("Distribution of Null Hypothesis and Treatment Loss")
plt.ylabel("Frequency")
plt.xlabel("Hamming Loss")

In [ ]:
# product a number of files
# 1: just based on matched files (results.csv)
# 2. balanced to eliminate overrepresentation (sampled_results.csv)
# 3. exact match (exact)
# 4. exact plus single instance 


In [ ]:
results[results.distance == 0].shape[0] / results.shape[0]

In [ ]:
results.head()

In [ ]:
br = pd.DataFrame(results[results.distance == 0 ]['test_ix'].value_counts()).reset_index()
results.index = results['test_ix']
sampled_exact_results = results.loc[list(br[br.test_ix ==1]['index'])]

In [ ]:
sampled_exact_results.head()

In [ ]:
results[results.distance == 0].to_csv('exact.csv', index=False)
sampled_exact_results.to_csv('sample_exact.csv', index=False)
sampled_results.to_csv('sample_results.csv', index=False)

In [ ]:
# diff.hist(bins=100)

In [ ]:
results.to_csv('results.csv', index=False)

In [ ]:
results.distance.hist(bins=100)

In [ ]:
# save covariates for each test and control

In [ ]:
# take difference 
# average difference for bias / variance diagnostics

In [ ]:
# find ways to pull data that produce different diagnostics 
# use best 